# H5 · Business environment: do credit and institutions go with deeper markets and more new businesses?

> **Hypothesis H5:** Countries with deeper credit systems have deeper stock markets, and countries with stronger institutions have more new businesses.

| | |
|---|---|
| **Why we ask** | Company profits are weakly tied to the country, but the *business environment itself* (how many firms start, how big the stock market is) may be where country conditions really show. This needs no company financials, so it covers more countries and gives an independent view. |
| **Prediction** | **Positive** associations: private credit (% of GDP) with stock-market value (% of GDP); institutions index with new business density. |
| **Data** | World Bank indicators, 2019–2023 country averages. Stock-market value is available for 22 countries (20 with credit data); new business density for 33. Eurostat business births (EU, 21 countries) for replication. |
| **Primary test (fixed before looking at the result)** | **Spearman ρ between private credit (% GDP) and stock-market value (% GDP)** across countries, with a bootstrap interval. (The institutions → new business density link is a secondary test.) |
| **Verdict rule** | *Supported* if the effect has the predicted sign, the wild-bootstrap p < 0.05, and the sign holds in at least 80% of the robustness samples. *Partly supported* if it has the predicted sign and p < 0.10 (or p < 0.05 but robustness is below 80%). *Contradicted* if significantly the opposite way. Otherwise *not supported*. *For H5 the 'effect' is a correlation and the p-value is the Spearman p-value; the robustness samples are leave-one-country-out, other time windows and exclusions of financial hubs.* |

**Honest framing.** Several hypotheses in this series (H1 to H3) were suggested by an exploratory scan of the same data, so their p-values are optimistic and a pass is a lead, not a confirmation. Where possible the result is **replicated on ESEF**, an independent source (which still shares some companies with Yahoo). All hypotheses are corrected for multiple testing together in `1.0_hypotheses.ipynb`. Everything is **association, not causation**.

In [1]:
import sys, pathlib, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src" / "analysis"))
import prep, utils
from utils import CONTROLS, fit_cluster, wild_cluster, spearman, boot_ci, partial_rho, bh, scan

con = prep.connect()
q = lambda sql: con.sql(sql).df()
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 120, "display.float_format", "{:,.3f}".format)
px.defaults.template = "plotly_white"

country = prep.load_country(con)
dim, profile, profile_all, panel = country["dim"], country["profile"], country["profile_all"], country["panel"]
feat_cols = country["feat_cols"]
fy = prep.load_firms(con, country)               # Yahoo company-years (primary sample)
names = dim.name
MIN_FIRMS = prep.MIN_FIRMS
print(f"{len(dim)} countries | {len(feat_cols)} country features | {len(fy):,} Yahoo company-years of {fy.company_id.nunique():,} companies")

SAMPLES = {
 "all firms": lambda d: d,
 "excluding financials & real estate": lambda d: d[~d.sector.isin(["Financial Services", "Real Estate"])],
 "excluding small-sample countries": lambda d: d[~d.iso3.isin(dim.index[dim.small_firm_sample])],
 "excluding USA, China, Japan": lambda d: d[~d.iso3.isin(["USA", "CHN", "JPN"])],
 "fiscal years 2023 and later": lambda d: d[d.fiscal_year >= 2023],
 "EU countries only": lambda d: d[d.iso3.isin(dim.index[dim.is_eu])],
 "non-EU countries only": lambda d: d[~d.iso3.isin(dim.index[dim.is_eu])],
}
def robustness(focal, outcome, expected_sign, B=499, extra=""):
    """Re-estimate the primary test in each sample. Returns a table and the share of samples with the expected sign."""
    rows = []
    for name, flt in SAMPLES.items():
        r = wild_cluster(focal, flt(fy), outcome=outcome, extra=extra, B=B)
        rows.append({"sample": name, "coef": r["coef"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"],
                     "countries": r["clusters"], "firm-years": r["n"]})
    t = pd.DataFrame(rows).set_index("sample")
    same = (np.sign(t.coef) == expected_sign) if expected_sign else pd.Series(True, index=t.index)
    return t.assign(**{"expected sign": same}), float(same.mean())

34 countries | 35 country features | 10,220 Yahoo company-years of 2,590 companies


## 1 · The outcomes
Three country-level outcomes (2019–2023 averages) and how many countries each one covers.

| Outcome | Meaning |
|---|---|
| **New business density** | New limited-liability companies registered per 1,000 people aged 15–64 |
| **Stock-market value (% of GDP)** | Total market value of listed domestic companies relative to the economy |
| **Listed companies** (log) | Number of domestically listed companies |

In [2]:
OUT = {"new_business_density": "new businesses per 1,000 people", "market_cap_gdp": "stock market value (% GDP)", "listed_companies": "listed companies (log)"}
lv = profile_all[list(OUT)].join(country["inst_idx"].rename("institutions index")).join(country["dev_idx"].rename("development index")).join(dim[["name", "region"]])
print({OUT[k]: int(lv[k].notna().sum()) for k in OUT}, "countries with a value")
lv.sort_values("market_cap_gdp")[["name", "new_business_density", "market_cap_gdp", "listed_companies", "institutions index"]].round(2)

{'new businesses per 1,000 people': 33, 'stock market value (% GDP)': 22, 'listed companies (log)': 22} countries with a value


,name,new_business_density,market_cap_gdp,listed_companies,institutions index
iso3,,,,,
CZE,Czechia,4.080,10.660,3.050,0.200
ROU,Romania,7.910,11.110,4.400,-0.870
HUN,Hungary,4.360,17.310,3.960,-0.770
POL,Poland,2.010,26.070,6.660,-0.490
GRC,Greece,2.160,28.040,5.080,-0.670
AUT,Austria,0.980,29.440,4.230,0.750
MEX,Mexico,0.890,33.420,4.910,-2.100
IDN,Indonesia,0.610,48.790,6.650,-1.540
DEU,Germany,1.420,52.230,6.100,0.820


## 2 · Primary test: credit and stock-market depth
Bank and other credit to firms and households (% of GDP) against the stock-market value (% of GDP). ρ is a rank correlation, the interval a bootstrap over countries (2,000 resamples), and the partial ρ controls for GDP per person, to see whether this is more than 'rich countries have everything'.

In [3]:
x_c, y_c = "private_credit_gdp", "market_cap_gdp"
pair = profile_all[[x_c, y_c, "gdp_per_capita_ppp"]].join(dim[["name", "region"]]).dropna(subset=[x_c, y_c])
rho, p_rho, n_rho = spearman(pair[x_c], pair[y_c])
lo, hi = boot_ci(pair[x_c], pair[y_c])
part = partial_rho(pair[x_c], pair[y_c], pair.gdp_per_capita_ppp)
print(f"n = {n_rho} countries | Spearman ρ = {rho:+.2f} (95% bootstrap CI {lo:+.2f} to {hi:+.2f}), p = {p_rho:.4f} | partial ρ controlling for GDP per person = {part:+.2f}")
fig = px.scatter(pair.reset_index(), x=x_c, y=y_c, color="region", text="iso3", title=f"Stock-market value vs private credit (ρ = {rho:+.2f}, n = {n_rho})",
                 labels={x_c: "private credit (% GDP)", y_c: "stock market value (% GDP)"})
b, a = np.polyfit(pair[x_c], pair[y_c], 1); xs = np.array([pair[x_c].min(), pair[x_c].max()])
fig.add_trace(go.Scatter(x=xs, y=a + b * xs, mode="lines", line=dict(color="grey", dash="dot"), showlegend=False))
fig.update_traces(textposition="top center", marker_size=9, selector=dict(mode="markers+text")); fig.update_layout(height=500); fig.show()

n = 20 countries | Spearman ρ = +0.77 (95% bootstrap CI +0.46 to +0.93), p = 0.0001 | partial ρ controlling for GDP per person = +0.85


### Secondary: institutions and new businesses

In [4]:
sc = lv[["institutions index", "new_business_density", "development index"]].dropna()
rho2, p2, n2 = spearman(sc["institutions index"], sc.new_business_density)
lo2, hi2 = boot_ci(sc["institutions index"], sc.new_business_density)
part2 = partial_rho(sc["institutions index"], sc.new_business_density, sc["development index"])
print(f"Institutions → new business density: n = {n2}, ρ = {rho2:+.2f} (95% CI {lo2:+.2f} to {hi2:+.2f}), p = {p2:.3f}; partial ρ controlling for development = {part2:+.2f}")
s = scan(profile.loc[lv.new_business_density.dropna().index], lv.new_business_density.dropna(), feat_cols)
print(f"Of all 35 country features, {(s.p < 0.05).sum()} have p < 0.05 for new business density ({0.05 * len(s):.1f} expected by chance); {(s.q < 0.10).sum()} have q < 0.10.")
s.head(8)

Institutions → new business density: n = 33, ρ = +0.27 (95% CI -0.09 to +0.59), p = 0.122; partial ρ controlling for development = -0.18
Of all 35 country features, 6 have p < 0.05 for new business density (1.8 expected by chance); 0 have q < 0.10.


,rho,p,n,q
feature,,,,
population,-0.436,0.011,33,0.264
gdp_usd,-0.378,0.030,33,0.264
corruption_control,0.368,0.035,33,0.264
internet_users,0.366,0.036,33,0.264
patents_residents,-0.354,0.044,33,0.264
gov_debt_gdp,-0.351,0.045,33,0.264
regulatory_quality,0.329,0.061,33,0.283
manufacturing_share,-0.325,0.065,33,0.283


### Do several features together explain stock-market depth?
Only 20 countries, so the model is deliberately small: credit, institutions and trade openness (standardised), robust errors. It checks whether credit still matters once institutions and openness are in.

In [5]:
md_ = profile_all[[x_c, y_c, "trade_gdp"]].join(country["inst_idx"].rename("inst")).dropna()
for c in [x_c, "trade_gdp"]:
    md_[c + "_z"] = (md_[c] - md_[c].mean()) / md_[c].std()
rows = []
for lab, f in {"credit only": f"{y_c} ~ {x_c}_z", "+ institutions": f"{y_c} ~ {x_c}_z + inst", "+ institutions + openness": f"{y_c} ~ {x_c}_z + inst + trade_gdp_z"}.items():
    m = smf.ols(f, md_).fit(cov_type="HC3")
    rows.append({"model": lab, "credit coef (pp GDP per +1 SD)": m.params[x_c + "_z"], "p": m.pvalues[x_c + "_z"], "institutions coef": m.params.get("inst", np.nan), "p (inst)": m.pvalues.get("inst", np.nan), "R²": m.rsquared, "n": int(m.nobs)})
pd.DataFrame(rows).set_index("model")

,credit coef (pp GDP per +1 SD),p,institutions coef,p (inst),R²,n
model,,,,,,
credit only,36.396,0.000,NaN,NaN,0.309,20
+ institutions,42.718,0.066,-15.654,0.511,0.362,20
+ institutions + openness,41.099,0.186,-13.510,0.667,0.364,20


## 3 · Robustness
The credit → stock-market link re-estimated by leaving out each country in turn, in earlier time windows, without financial hubs, and with log values. The robustness share is the share of these variants with a positive ρ.

In [6]:
def window_profile(y0, y1):
    w = panel[panel.year.between(y0, y1)]
    return w.groupby("iso3")[[x_c, y_c, "gdp_per_capita_ppp"]].agg(lambda s: s.mean() if s.notna().sum() >= 3 else np.nan)
variants = {}
for y0, y1 in [(2012, 2016), (2014, 2018), (2016, 2020), (2019, 2023)]:
    wp = window_profile(y0, y1).dropna(subset=[x_c, y_c]); variants[f"window {y0}–{y1}"] = spearman(wp[x_c], wp[y_c])
ex = pair.drop(index=[i for i in ["LUX", "SGP", "IRL"] if i in pair.index]); variants["without Luxembourg, Singapore, Ireland"] = spearman(ex[x_c], ex[y_c])
variants["without USA, China, Japan"] = spearman(pair.drop(index=[i for i in ["USA", "CHN", "JPN"] if i in pair.index])[x_c], pair.drop(index=[i for i in ["USA", "CHN", "JPN"] if i in pair.index])[y_c])
variants["Pearson on logs"] = (stats.pearsonr(np.log(pair[x_c]), np.log(pair[y_c]))[0], stats.pearsonr(np.log(pair[x_c]), np.log(pair[y_c]))[1], len(pair))
loo = {i: spearman(pair.drop(index=i)[x_c], pair.drop(index=i)[y_c])[0] for i in pair.index}
rob = pd.DataFrame([{"variant": k, "ρ": v[0], "p": v[1], "n": v[2]} for k, v in variants.items()]).set_index("variant")
loo_s = pd.Series(loo)
all_rho = pd.concat([rob["ρ"], loo_s.rename(None).rename(lambda i: f"leave out {i}")])
rob_share = float((all_rho > 0).mean())
print(f"Leave-one-out ρ ranges from {loo_s.min():+.2f} ({loo_s.idxmin()}) to {loo_s.max():+.2f} ({loo_s.idxmax()}). Positive ρ in {rob_share:.0%} of {len(all_rho)} variants.")
rob

Leave-one-out ρ ranges from +0.74 (ROU) to +0.85 (IND). Positive ρ in 100% of 27 variants.


,ρ,p,n
variant,,,
window 2012–2016,0.525,0.006,26
window 2014–2018,0.631,0.001,25
window 2016–2020,0.732,0.000,25
window 2019–2023,0.768,0.000,20
"without Luxembourg, Singapore, Ireland",0.761,0.000,19
"without USA, China, Japan",0.713,0.001,17
Pearson on logs,0.708,0.000,20


## 4 · Replication: business births in the EU (Eurostat)
An independent source for the *new businesses* half of the hypothesis: the **business birth rate** (new enterprises as % of active enterprises, 2021–2024 average) for the 21 EU countries, against the institutions index.

In [7]:
birth = q("SELECT iso3, AVG(value) AS birth_rate FROM fact_country_year WHERE indicator_id LIKE 'ESTAT:bd_size:%ENT_BRTHR_PC%' GROUP BY 1").set_index("iso3")
eb = birth.join(country["inst_idx"].rename("inst")).join(country["dev_idx"].rename("dev")).dropna()
r3, p3, n3 = spearman(eb.inst, eb.birth_rate)
lo3, hi3 = boot_ci(eb.inst, eb.birth_rate)
print(f"EU business birth rate vs institutions index: n = {n3}, ρ = {r3:+.2f} (95% CI {lo3:+.2f} to {hi3:+.2f}), p = {p3:.3f}; partial ρ controlling for development = {partial_rho(eb.inst, eb.birth_rate, eb.dev):+.2f}")
px.scatter(eb.join(dim[["name"]]).reset_index(), x="inst", y="birth_rate", text="iso3", title=f"EU business birth rate vs institutions index (ρ = {r3:+.2f}, n = {n3})", labels={"inst": "institutions index (SD)", "birth_rate": "business birth rate (%)"}).update_traces(textposition="top center", marker_size=9).update_layout(height=440)

EU business birth rate vs institutions index: n = 21, ρ = -0.31 (95% CI -0.61 to +0.10), p = 0.173; partial ρ controlling for development = +0.10


## 5 · Verdict

In [8]:
v = utils.verdict(rho, p_rho, +1, rob_share, int(n_rho))
lines = [
 f"Primary: credit vs stock-market value ρ = {rho:+.2f} (95% CI {lo:+.2f} to {hi:+.2f}), p = {p_rho:.4f}, n = {n_rho} countries; partial ρ given GDP per person {part:+.2f}.",
 f"Robustness: positive in {rob_share:.0%} of {len(all_rho)} variants (leave-one-out range {loo_s.min():+.2f} to {loo_s.max():+.2f}).",
 f"Secondary: institutions → new business density ρ = {rho2:+.2f} (p = {p2:.3f}, n = {n2}); EU replication with business birth rate ρ = {r3:+.2f} (p = {p3:.3f}, n = {n3}).",
 f"VERDICT H5: {v.upper()}",
]
print("\n".join(lines))
utils.save_result("H5", id="H5", title="Business environment", statement="Deeper credit systems go with deeper stock markets.",
                  expected_sign=1, effect=float(rho), ci_low=float(lo), ci_high=float(hi), effect_unit="Spearman ρ, private credit vs stock-market value (% GDP)",
                  p_primary=float(p_rho), p_method="Spearman test (analytic)", n=int(n_rho), clusters=int(n_rho), robust_share=rob_share,
                  replication_coef=float(r3), replication_p=float(p3), replication_same_sign=bool(r3 > 0), verdict=v,
                  secondary_effect=float(rho2), secondary_p=float(p2), notebook="1.5_business_environment.ipynb", notes=lines)

Primary: credit vs stock-market value ρ = +0.77 (95% CI +0.46 to +0.93), p = 0.0001, n = 20 countries; partial ρ given GDP per person +0.85.
Robustness: positive in 100% of 27 variants (leave-one-out range +0.74 to +0.85).
Secondary: institutions → new business density ρ = +0.27 (p = 0.122, n = 33); EU replication with business birth rate ρ = -0.31 (p = 0.173, n = 21).
VERDICT H5: SUPPORTED


### Limits
- Stock-market value is available for only 20–22 countries, and a few financial hubs (Luxembourg, Singapore, Ireland) have unusual values; the robustness table shows what happens without them.
- Credit and stock-market depth both grow with development, so the link may be a shared dependence on wealth; the partial correlation controls for GDP per person but cannot rule out other common causes.
- The replication covers the *new businesses* half of the hypothesis (EU only); the credit-and-markets half has no independent source here.
- Association, not causation.

In [9]:
con.close()